<a href="https://colab.research.google.com/github/ChikleNeha/deep_rl_course/blob/main/Unit1_lunar_lander2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install system dependencies for Box2D and Video Recording
!apt-get update && apt-get install -y swig python3-dev ffmpeg

# 2. Install the modern RL libraries
# We use 'gymnasium[box2d]' (the successor to gym) and SB3 v2.3+
!pip install "gymnasium[box2d]>=1.0.0"
!pip install "stable-baselines3[extra]>=2.3.2"
!pip install shimmy  # Required for compatibility between older envs and new SB3
!pip install huggingface_sb3  # To push your model to the Hub later

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# 1. Create the environment
# We use render_mode="rgb_array" so we can record videos later
env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = Monitor(env) # Good practice for tracking statistics

# 2. Reset the environment (Note: Gymnasium returns two values now)
observation, info = env.reset()

print(f"✅ Environment initialized! Observation shape: {observation.shape}")

In [ ]:
# trying again
from stable_baselines3 import PPO

# 1. Re-initialize with "Expert" Hyperparameters for Python 3.12
# These are optimized to help the agent 'commit' to the landing
model = PPO(
    policy = "MlpPolicy",
    env = env,
    learning_rate = 0.0003, # Standard but effective
    n_steps = 2048,         # Larger buffer to see the whole flight
    batch_size = 64,        # Stable updates
    n_epochs = 10,          # More optimization per update
    gamma = 0.999,          # Focus on the very end of the episode (the landing)
    gae_lambda = 0.98,
    ent_coef = 0.01,        # Encourages exploration (prevents 'stopping midway')
    verbose = 1
)

# 2. Train for a solid 500k steps
# This is usually the 'magic number' for a perfect LunarLander
print("🚀 Starting the Master Training (500k steps)...")
model.learn(total_timesteps=500000, progress_bar=True)

# 3. Save the masterpiece
model.save("ppo-LunarLander-v3-expert")

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

# Evaluate the new version
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True)

print(f"🏆 Final Model Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

if mean_reward > 200:
    print("⭐ High Score! Your agent has mastered the landing.")
else:
    print("📈 Improving! It might need a bit more time or a higher learning rate.")

In [ ]:
import gymnasium as gym
import imageio
import numpy as np
from IPython.display import Image, display

def record_and_display_video(env_id, model, video_length=1000, out_file="lunar_lander.gif"):
    eval_env = gym.make(env_id, render_mode="rgb_array")
    images = []
    obs, info = eval_env.reset()

    for i in range(video_length):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        images.append(eval_env.render())
        if terminated or truncated:
            break

    eval_env.close()
    imageio.mimsave(out_file, [np.array(img) for img in images], fps=30)
    display(Image(open(out_file, 'rb').read()))

In [ ]:
# This uses the function we defined in the previous step
print("🎬 Recording the updated landing...")
record_and_display_video("LunarLander-v3", model, out_file="final_victory.gif")